[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/04_Finetuning_LowCompute/03_adapter_methods/03_adapter_methods.ipynb)

# 03. Adapter Methods: Beyond LoRA

**This notebook covers:**
- Bottleneck Adapters — insert small layers
- Prefix Tuning — learn virtual tokens
- Prompt Tuning — simplest PEFT method
- IA3 — rescale activations
- Side-by-side comparison: which method when?

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/04_Finetuning_LowCompute/03_adapter_methods")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# Overview diagram of all PEFT methods

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Parameter-Efficient Fine-Tuning Methods', fontsize=18, fontweight='bold')

methods = [
    ('LoRA', 'Add low-rank\nmatrices A, B\nto weight W', '#E74C3C',
     'Trainable: A (d->r) + B (r->d)\nFrozen: W'),
    ('Adapter', 'Insert bottleneck\nlayers between\ntransformer blocks', '#3498DB',
     'Trainable: Down + Up projections\nFrozen: original layers'),
    ('Prefix Tuning', 'Prepend learnable\nvirtual tokens\nto K and V', '#2ECC71',
     'Trainable: prefix vectors\nFrozen: all model params'),
    ('Prompt Tuning', 'Prepend learnable\nembeddings to\ninput sequence', '#F39C12',
     'Trainable: soft prompt embeds\nFrozen: entire model'),
    ('IA3', 'Learn scaling\nvectors for\nK, V, and FFN', '#9B59B6',
     'Trainable: 3 vectors per layer\nFrozen: all weights'),
    ('BitFit', 'Only train\nbias terms\nin the model', '#1ABC9C',
     'Trainable: bias parameters\nFrozen: all weights'),
]

for ax, (name, desc, color, details) in zip(axes.flat, methods):
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 6)
    ax.axis('off')
    draw_architecture_block(ax, 3, 4.5, 4, 1.5, desc, color, fontsize=9)
    ax.set_title(name, fontsize=14, fontweight='bold', color=color)
    ax.text(3, 2.2, details, ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../assets/peft_methods_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 1. Bottleneck Adapter

**Idea:** Insert a small trainable module after each transformer sublayer.

```
x → [Frozen Attention] → + → [Adapter: Down→ReLU→Up] → + → x_out
                          ↑                              ↑
                        residual                      residual
```

### Bottleneck Adapter Math

$$h = h + f(hW_{\text{down}})W_{\text{up}}$$

where $W_{\text{down}} \in \mathbb{R}^{d \times m}$, $W_{\text{up}} \in \mathbb{R}^{m \times d}$, and $m \ll d$.

**Parameters per adapter:** $2dm + d + m$ (including biases). For $d=768$, $m=64$: **98,368 params** vs 589,824 for a full layer (**16.7%**).

Zero-initialization of $W_{\text{up}}$ ensures the adapter starts as identity.

In [ ]:
class BottleneckAdapter(nn.Module):
    """Adapter layer: down-project -> activation -> up-project + residual."""
    def __init__(self, dim, bottleneck_dim=64):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck_dim)
        self.up = nn.Linear(bottleneck_dim, dim)
        self.act = nn.GELU()
        # Initialize up-projection to near-zero so adapter starts as identity
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


# Demo
adapter = BottleneckAdapter(dim=768, bottleneck_dim=64)
x = torch.randn(2, 10, 768)
out = adapter(x)

adapter_params = sum(p.numel() for p in adapter.parameters())
full_layer_params = 768 * 768
print(f"Adapter params:     {adapter_params:,}")
print(f"Full layer params:  {full_layer_params:,}")
print(f"Ratio:              {adapter_params/full_layer_params*100:.1f}%")
print(f"Output shape:       {out.shape}")

## 2. Prefix Tuning

**Idea:** Learn virtual prefix tokens that are prepended to Keys and Values in every attention layer.
The model "sees" these extra tokens and learns to condition on them.

### Prefix Tuning Reparameterization

Direct optimization of prefix vectors is **unstable**. Solution: use an MLP to reparameterize:

$$P_k^{(l)} = \text{MLP}_\theta(E_k)$$

where $E_k$ are small learnable embeddings.

During training, optimize MLP parameters. After training, discard the MLP and keep the generated prefix vectors. This stabilizes optimization.

In [ ]:
class PrefixTuning(nn.Module):
    """Learn prefix vectors prepended to keys and values."""
    def __init__(self, dim, n_prefix=10, n_layers=4):
        super().__init__()
        self.n_prefix = n_prefix
        # Separate prefix for each layer's K and V
        self.prefix_k = nn.ParameterList([
            nn.Parameter(torch.randn(1, n_prefix, dim) * 0.01)
            for _ in range(n_layers)
        ])
        self.prefix_v = nn.ParameterList([
            nn.Parameter(torch.randn(1, n_prefix, dim) * 0.01)
            for _ in range(n_layers)
        ])

    def get_prefix(self, layer_idx, batch_size):
        pk = self.prefix_k[layer_idx].expand(batch_size, -1, -1)
        pv = self.prefix_v[layer_idx].expand(batch_size, -1, -1)
        return pk, pv


prefix = PrefixTuning(dim=768, n_prefix=10, n_layers=4)
prefix_params = sum(p.numel() for p in prefix.parameters())
print(f"Prefix params:  {prefix_params:,}")
print(f"That's {prefix_params/(768*768*4)*100:.2f}% of 4 attention layers")

# Visualize: how prefix works
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('Prefix Tuning: Virtual Tokens Prepended to K,V', fontsize=14, fontweight='bold')

# Prefix tokens
for i in range(4):
    draw_architecture_block(ax, 1 + i*1.2, 2, 1, 0.6, f'P{i}', '#2ECC71')
    draw_architecture_block(ax, 1 + i*1.2, 1, 1, 0.6, f'P{i}', '#2ECC71')

# Real tokens
real_tokens = ['[CLS]', 'a', 'cat', 'photo']
for i, tok in enumerate(real_tokens):
    draw_architecture_block(ax, 6 + i*1.5, 2, 1.2, 0.6, tok, '#3498DB')
    draw_architecture_block(ax, 6 + i*1.5, 1, 1.2, 0.6, tok, '#3498DB')

ax.text(0.2, 2, 'K:', fontsize=12, fontweight='bold', va='center')
ax.text(0.2, 1, 'V:', fontsize=12, fontweight='bold', va='center')
ax.text(2.5, 2.7, 'Learned (trainable)', fontsize=9, color='#2ECC71', ha='center')
ax.text(8.5, 2.7, 'From input (frozen model)', fontsize=9, color='#3498DB', ha='center')

plt.tight_layout()
plt.show()

## 3. Prompt Tuning

**Simplest PEFT method.** Learn soft embeddings prepended to the input. The entire model stays frozen.

### Prompt Tuning Scaling Laws

Lester et al. (2021) showed that prompt tuning performance **scales with model size**:

- **T5-Small (60M):** prompt tuning ≪ full finetuning
- **T5-Base (220M):** gap narrows
- **T5-XXL (11B):** prompt tuning ≈ full finetuning!

With just **20 soft tokens** ($20 \times d$ parameters), you can match full finetuning on large models. This is because larger models have more capacity to interpret soft prompts.

In [ ]:
class PromptTuning(nn.Module):
    """Prepend learnable soft prompt embeddings to input."""
    def __init__(self, dim, n_prompt_tokens=20):
        super().__init__()
        self.soft_prompt = nn.Parameter(torch.randn(1, n_prompt_tokens, dim) * 0.01)

    def forward(self, input_embeds):
        B = input_embeds.shape[0]
        prompt = self.soft_prompt.expand(B, -1, -1)
        return torch.cat([prompt, input_embeds], dim=1)


pt = PromptTuning(dim=768, n_prompt_tokens=20)
x = torch.randn(2, 10, 768)
out = pt(x)
print(f"Input:  {x.shape}  (batch=2, seq=10, dim=768)")
print(f"Output: {out.shape}  (batch=2, seq=30, dim=768)  <- 20 prompt + 10 input")
print(f"Trainable params: {sum(p.numel() for p in pt.parameters()):,}")

## 4. IA3 (Infused Adapter by Inhibiting and Amplifying Inner Activations)

**Idea:** Learn element-wise scaling vectors for K, V, and FFN outputs. Even fewer parameters than LoRA.

### IA3 Math

$$h_k = l_k \odot (xW^K), \quad h_v = l_v \odot (xW^V), \quad h_{\text{ff}} = l_{\text{ff}} \odot f(xW_1)W_2$$

Where $l_k, l_v, l_{\text{ff}} \in \mathbb{R}^d$ are learnable vectors (initialized to ones).

**Total params per layer:** just $3d$. For $d=768$: **2,304 params** vs 589,824 for a full layer (**0.4%**).

**Key insight:** rescaling is sufficient for task adaptation because pretrained features are already good — they just need to be re-weighted.

In [ ]:
class IA3Layer(nn.Module):
    """IA3: learned rescaling of keys, values, and FFN."""
    def __init__(self, dim):
        super().__init__()
        self.scale_k = nn.Parameter(torch.ones(dim))
        self.scale_v = nn.Parameter(torch.ones(dim))
        self.scale_ff = nn.Parameter(torch.ones(dim))

    def rescale_kv(self, keys, values):
        return keys * self.scale_k, values * self.scale_v

    def rescale_ff(self, ff_output):
        return ff_output * self.scale_ff


ia3 = IA3Layer(dim=768)
print(f"IA3 params per layer: {sum(p.numel() for p in ia3.parameters()):,}")
print(f"That's just 3 vectors of size {768}!")

In [ ]:
# Grand comparison chart

dim = 768
methods_data = {
    'Full FT':      dim * dim,
    'LoRA (r=8)':   2 * dim * 8,
    'Adapter (64)': dim * 64 * 2 + 64 + dim,
    'Prefix (10)':  2 * 10 * dim,
    'Prompt (20)':  20 * dim,
    'IA3':          3 * dim,
    'BitFit':       dim,
}

fig, ax = plt.subplots(figsize=(12, 6))
names = list(methods_data.keys())
vals = [v/1000 for v in methods_data.values()]
colors = ['#34495E', '#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']

bars = ax.barh(names, vals, color=colors, alpha=0.85)
for bar, (name, v) in zip(bars, methods_data.items()):
    pct = v / (dim*dim) * 100
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{v:,} ({pct:.1f}%)', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Trainable Parameters (K) per layer', fontsize=12)
ax.set_title(f'PEFT Methods Comparison (dim={dim})', fontsize=14, fontweight='bold')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('../assets/peft_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Decision Guide

| Method | Best For | Params | Quality | Inference Cost |
|--------|----------|--------|---------|--------|
| **LoRA** | General default | Low | High | Same (merge) |
| **QLoRA** | Large models, tiny GPU | Lowest mem | High | Slight overhead |
| **Adapter** | Multi-task (swap per task) | Medium | High | Slight overhead |
| **Prefix** | NLG tasks | Very low | Good | Same |
| **Prompt** | Classification, huge models | Minimal | Moderate | Same |
| **IA3** | Fastest training | Minimal | Good | Same |

**For low compute:** Start with **QLoRA (r=8)**, try **IA3** if you need even less.

---
**Next:** `04_finetune_clip_custom_data.ipynb` - Put it all together!